# V2 Pseudo-Label Coreset

Robust pseudo-labeling pipeline for `coreset.csv` with:
- Safe context budgeting (never truncates system/few-shot headers)
- `matched_paras` list-vs-string normalisation
- Deterministic generation (`do_sample=False`)
- Resume/done-set support
- Two-pass error recovery
- Submission exporter (per-doc JSON + ZIP)

In [ ]:
# ==== Cell 0: Imports & Path Constants ====
import json
import re
import os
import zipfile
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Paths (edit to match your environment) ----
CSV_PATH      = Path("/root/autodl-fs/coreset.csv")          # input coreset
TAGS_PATH     = Path("/root/autodl-fs/tags.json")            # optional tag whitelist JSON
MODEL_PATH    = "/root/autodl-fs/models/Qwen2.5-7B-Instruct" # or any HF model id
OUT_DIR       = Path("/root/autodl-fs/v2_output")
OUT_JSONL     = OUT_DIR / "pseudo_labels_v2.jsonl"
ERRORS_JSONL  = OUT_DIR / "errors_v2.jsonl"
SUBMIT_DIR    = OUT_DIR / "submission_v2"
ZIP_PATH      = OUT_DIR / "submission_v2.zip"

OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMIT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", OUT_DIR)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ==== Cell 1: V2 Constants & Trigger-Word Lists ====
CTX_N          = 3    # context window (preceding paragraphs fed to model)
MAX_NEW_TOKENS = 220  # sufficient for JSON + think template
TOPK_LINKS     = 3    # max matched_paras links per paragraph

VALID_REL  = {"contradictive", "supporting", "complemental", "modifying"}
VALID_TYPE = {"preambular", "operative"}

# French trigger words
TRIG_PREAMBULAR = [
    "Rappelant", "Réaffirmant", "Considérant", "Soulignant", "Notant",
    "Prenant note", "Ayant à l'esprit", "Reconnaissant", "Conscient",
    "Alarmé", "Préoccupé"
]
TRIG_OPERATIVE = [
    "Décide", "Invite", "Prie", "Demande", "Encourage", "Recommande",
    "Charge", "Adopte", "Appelle", "Exhorte", "Souhaite"
]
TRIG_ALL = sorted(set(TRIG_PREAMBULAR + TRIG_OPERATIVE), key=len, reverse=True)

# Boundary-check hints
MODIFY_HINT = [
    "sous réserve", "à condition", "toutefois", "nonobstant",
    "à l'exception", "modifie", "remplace", "abroge", "révise", "amende"
]
ADD_HINT = [
    "en outre", "par ailleurs", "de plus", "également",
    "notamment", "y compris"
]

# Relation priority (used when a list needs to be collapsed to one string)
REL_PRI = ["modifying", "contradictive", "supporting", "complemental"]

# Tag alias table (add entries as needed)
ALIAS: dict = {}

print("V2 constants loaded.")

In [ ]:
# ==== Cell 2: Load Tag Whitelist ====
# If TAGS_PATH exists, load it; otherwise use an empty whitelist (all tags pass).
if TAGS_PATH.exists():
    with open(TAGS_PATH, "r", encoding="utf-8") as _f:
        valid_codes: set = set(json.load(_f))
    print(f"Loaded {len(valid_codes)} valid tags from {TAGS_PATH}")
else:
    valid_codes: set = set()
    print("No tags.json found – all tags accepted (no whitelist filtering).")

In [ ]:
# ==== Cell 3: System Prompt & User Prompt Builder ====
SYSTEM_PROMPT = """You are an expert in argument mining for UN/UNESCO French documents.

OUTPUT RULE:
Return ONLY one valid JSON object. No markdown. No extra text.

Required fields (must all exist):
- para_number: int
- type: "preambular" or "operative"
- tags: string[] (from whitelist; output [] if none fit)
- matched_paras: dict (keys = context pids as strings; values = one of "supporting"|"contradictive"|"complemental"|"modifying"; output {} if no clear link)
- think: strictly follow template: "Triggers: [t1, t2]. Check: modifies_scope=[True/False], adds_parallel_info=[True/False]."
- evidence_triggers: string[]
- boundary_check: {"modifies_scope": bool, "adds_parallel_info": bool}
- model_confidence: "A"|"B"|"C"
- uncertainty_flag: string ("" if none)

Relation guidance:
- modifying: changes scope/conditions/obligations, adds exceptions, amends an earlier rule.
- complemental: adds parallel info/examples WITHOUT changing scope/conditions.
Tie-breaker: choose modifying ONLY when scope/conditions explicitly change; otherwise complemental.
"""


def build_user_prompt(
    doc_id: str,
    pid,
    text_fr: str,
    prev_rows: list,
    weak_role_hint: str | None = None,
) -> str:
    """Build the per-paragraph user prompt.

    Parameters
    ----------
    doc_id       : document identifier
    pid          : target paragraph id (int or str)
    text_fr      : French text of the target paragraph
    prev_rows    : list of dicts with keys 'pid' and 'text_fr' (context paragraphs)
    weak_role_hint : optional hint ('preambular_anchor' | 'operative_anchor')
    """
    ctx_lines = [
        f"<P{int(r['pid'])}> {str(r['text_fr'])}" for r in prev_rows
    ]
    code_str = ", ".join(sorted(valid_codes)) if valid_codes else "(no whitelist)"
    hint = (
        weak_role_hint
        if weak_role_hint in ("preambular_anchor", "operative_anchor")
        else "unknown"
    )

    return f"""Task: annotate the TARGET paragraph using the provided CONTEXT paragraphs.

doc_id: {doc_id}
weak_role_hint: {hint}

CONTEXT:
{chr(10).join(ctx_lines) if ctx_lines else "(no context)"}

TARGET:
<P{int(pid)}> {text_fr} </P{int(pid)}>

TAG WHITELIST:
[{code_str}]

Return ONLY one JSON object.""".strip()

In [ ]:
# ==== Cell 4: Evidence Trigger Extraction & Boundary-Check Helpers ====

def extract_triggers(text_fr: str) -> list:
    """Return up to 5 French trigger words found in *text_fr*."""
    t = text_fr.strip()
    hits: list = []
    for w in TRIG_ALL:
        if re.search(rf"(^|[\s,;:]){re.escape(w)}([\s,;:])", t):
            hits.append(w)
    # deduplicate while preserving order
    out: list = []
    seen: set = set()
    for x in hits:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out[:5]


def infer_boundary_check(text_fr: str) -> dict:
    """Rule-based boundary check using French hint phrases."""
    low = text_fr.lower()
    modifies = any(h in low for h in MODIFY_HINT)
    adds = any(h in low for h in ADD_HINT)
    return {"modifies_scope": bool(modifies), "adds_parallel_info": bool(adds)}


def format_think(evidence_triggers: list, boundary_check: dict) -> str:
    """Produce the structured English think template string."""
    et = evidence_triggers if evidence_triggers else []
    ms = boundary_check.get("modifies_scope", False)
    ap = boundary_check.get("adds_parallel_info", False)
    return f"Triggers: {et}. Check: modifies_scope={ms}, adds_parallel_info={ap}."


print("Trigger helpers loaded.")

In [ ]:
# ==== Cell 5: Safe Context Budgeting ====
# Ensures the full prompt fits within the model's context window by trimming
# *only* context paragraphs. System prompt and few-shot headers are never removed.

def safe_context_budget(
    tokenizer,
    system_prompt: str,
    user_prompt: str,
    max_model_len: int = 4096,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    """Return a (possibly trimmed) user_prompt that fits the budget.

    Strategy
    --------
    1. Measure fixed overhead: system prompt + TARGET block + TAG WHITELIST block.
    2. Allocate remaining tokens to the CONTEXT block.
    3. If CONTEXT is too long, drop the *oldest* context lines one at a time.
    """
    budget = max_model_len - max_new_tokens - 64  # 64 = chat-template overhead

    # Split user_prompt into CONTEXT section and the rest
    ctx_marker   = "CONTEXT:"
    target_marker = "TARGET:"

    if ctx_marker not in user_prompt or target_marker not in user_prompt:
        # Cannot split safely; return as-is and hope for the best
        return user_prompt

    before_ctx, rest = user_prompt.split(ctx_marker, 1)
    ctx_block, after_ctx = rest.split(target_marker, 1)

    ctx_lines = [ln for ln in ctx_block.strip().splitlines() if ln.strip()]

    fixed_text  = system_prompt + before_ctx + target_marker + after_ctx
    fixed_tokens = len(tokenizer.encode(fixed_text, add_special_tokens=False))

    remaining = budget - fixed_tokens

    # Trim from oldest (front) until the context fits
    while ctx_lines:
        ctx_text   = "\n".join(ctx_lines)
        ctx_tokens = len(tokenizer.encode(ctx_text, add_special_tokens=False))
        if ctx_tokens <= remaining:
            break
        ctx_lines.pop(0)  # remove oldest context paragraph

    ctx_section = "\n".join(ctx_lines) if ctx_lines else "(no context)"
    trimmed = (
        before_ctx
        + ctx_marker + "\n"
        + ctx_section + "\n\n"
        + target_marker
        + after_ctx
    )
    return trimmed


print("Safe context budgeting loaded.")

In [ ]:
# ==== Cell 6: normalize_pred – Robust Matched-Paras Normalisation ====

def flatten_rel_list(rels: list):
    """Collapse a list of relations to the highest-priority single relation."""
    if not rels:
        return None
    for r in REL_PRI:
        if r in rels:
            return r
    return rels[0]


def normalize_pred(
    obj: dict,
    pid,
    allowed_ctx_pids: set,
    text_fr: str,
    weak_role_hint: str = "unknown",
) -> dict:
    """Normalise a raw model-output dict into a clean annotation.

    Handles:
    - matched_paras values that are strings OR lists
    - Tags that are strings or lists, filtered through whitelist
    - Type prior from French trigger words
    - TopK link truncation
    - Rule-based evidence_triggers & boundary_check fallback
    - Enforced structured think template
    """
    default_type = "operative" if weak_role_hint == "operative_anchor" else "preambular"

    out = {
        "para_number": int(pid),
        "type": default_type,
        "tags": [],
        "matched_paras": {},
        "think": "",
        "evidence_triggers": [],
        "boundary_check": {"modifies_scope": False, "adds_parallel_info": False},
        "model_confidence": "C",
        "uncertainty_flag": "",
        "_invalid_tags": [],
    }

    # --- type ---
    tp = str(obj.get("type", default_type)).strip().lower()
    if tp not in VALID_TYPE:
        tp = default_type

    # type prior: if operative trigger found in text, prefer operative
    trig = extract_triggers(text_fr)
    if any(x in TRIG_OPERATIVE for x in trig):
        tp = "operative"
    out["type"] = tp

    # --- tags ---
    raw_tags = obj.get("tags", [])
    if isinstance(raw_tags, str):
        raw_tags = [raw_tags]
    fixed_tags: list = []
    invalid_tags: list = []
    for t in raw_tags if isinstance(raw_tags, list) else []:
        t = str(t).strip()
        t = ALIAS.get(t, t)
        if not valid_codes or t in valid_codes:
            fixed_tags.append(t)
        else:
            invalid_tags.append(t)
    out["tags"] = list(dict.fromkeys(fixed_tags))   # deduplicate, preserve order
    out["_invalid_tags"] = invalid_tags

    # --- matched_paras (keep as internal list for boundary checker) ---
    mp = obj.get("matched_paras", {})
    clean_mp: dict = {}
    if isinstance(mp, dict):
        for k, v in mp.items():
            k = str(k).strip().lstrip("P")          # accept "P3" or "3"
            if not k.isdigit():
                continue
            k_int = int(k)
            if k_int not in allowed_ctx_pids:
                continue
            # Normalise value: accept string OR list
            if isinstance(v, str):
                v = [v]
            elif not isinstance(v, list):
                continue
            rels = [
                str(x).strip().lower()
                for x in v
                if str(x).strip().lower() in VALID_REL
            ]
            rels = list(dict.fromkeys(rels))         # deduplicate, preserve order
            if rels:
                clean_mp[str(k_int)] = rels

    # TopK truncation – keep nearest pids first
    if len(clean_mp) > TOPK_LINKS:
        keys = sorted(clean_mp.keys(), key=lambda x: abs(int(x) - int(pid)))
        clean_mp = {k: clean_mp[k] for k in keys[:TOPK_LINKS]}
    out["matched_paras"] = clean_mp

    # --- evidence_triggers (prefer rule extraction as fallback) ---
    et = obj.get("evidence_triggers", [])
    if not isinstance(et, list) or len(et) == 0:
        et = trig
    out["evidence_triggers"] = et

    # --- boundary_check (merge model output with rule-based result) ---
    bc = obj.get("boundary_check", {})
    if not isinstance(bc, dict):
        bc = {}
    rule_bc = infer_boundary_check(text_fr)
    out["boundary_check"] = {
        "modifies_scope":    bool(bc.get("modifies_scope",    rule_bc["modifies_scope"])),
        "adds_parallel_info": bool(bc.get("adds_parallel_info", rule_bc["adds_parallel_info"])),
    }

    # --- think (always use structured English template) ---
    out["think"] = format_think(out["evidence_triggers"], out["boundary_check"])[:300]

    # --- confidence / uncertainty ---
    conf = str(obj.get("model_confidence", "C")).strip().upper()
    out["model_confidence"] = conf if conf in {"A", "B", "C"} else "C"
    out["uncertainty_flag"] = str(obj.get("uncertainty_flag", "")).strip()[:200]

    return out


print("normalize_pred loaded.")

In [ ]:
# ==== Cell 7: Boundary Consistency Checker (M2) ====

REVIEW_QUEUE: list = []


def boundary_consistency_check(ann: dict, doc_id: str, pid) -> dict:
    """Flag inconsistent boundary annotations and push them to REVIEW_QUEUE.

    Rules
    -----
    1. modifies_scope=True but no matched_paras with 'modifying'  → flag
    2. matched_paras has 'modifying' but modifies_scope=False      → flag
    3. Same pid has both 'complemental' and 'modifying'            → downgrade to modifying

    Side effects
    ------------
    Appends a dict to the module-level ``REVIEW_QUEUE`` list whenever one or
    more issues are detected.  The annotation dict also receives a
    ``_review_issues`` key listing the detected problems.
    """
    bc = ann.get("boundary_check", {})
    mp = ann.get("matched_paras", {})
    issues: list = []

    has_modifying = any(
        "modifying" in (v if isinstance(v, list) else [v])
        for v in mp.values()
    )

    if bc.get("modifies_scope") and not has_modifying:
        issues.append("modifies_scope=True but no modifying relation")

    if has_modifying and not bc.get("modifies_scope"):
        issues.append("modifying relation found but modifies_scope=False")

    # Rule 3: collapse complemental+modifying → modifying
    clean_mp: dict = {}
    for k, v in mp.items():
        rels = v if isinstance(v, list) else [v]
        if "complemental" in rels and "modifying" in rels:
            rels = [r for r in rels if r != "complemental"]
            issues.append(f"pid {k}: complemental+modifying → downgraded to modifying")
        clean_mp[k] = rels
    ann["matched_paras"] = clean_mp

    if issues:
        REVIEW_QUEUE.append({"doc_id": doc_id, "pid": pid, "issues": issues, "ann": ann})
        ann["_review_issues"] = issues

    return ann


print("Boundary consistency checker loaded. REVIEW_QUEUE cleared.")

In [ ]:
# ==== Cell 8: Load Model & Tokenizer ====
print(f"Loading model from: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

# Determine context window from model config (fallback 4096)
MAX_MODEL_LEN = getattr(model.config, "max_position_embeddings", 4096)
print(f"Model loaded. max_position_embeddings = {MAX_MODEL_LEN}")

In [ ]:
# ==== Cell 9: Load Coreset & Build Done-Set (Resume Support) ====

df = pd.read_csv(CSV_PATH).sort_values(["doc_id", "pid"]).reset_index(drop=True)
print(f"Coreset loaded: {len(df)} rows, {df['doc_id'].nunique()} documents")
print(df.dtypes)

# Load existing output to build done-set (enables interrupted-run resume)
done_set: set = set()  # elements are (doc_id, pid) tuples
if OUT_JSONL.exists():
    with open(OUT_JSONL, "r", encoding="utf-8") as _f:
        for _line in _f:
            if _line.strip():
                try:
                    _row = json.loads(_line)
                    done_set.add((_row["doc_id"], int(_row["pid"])))
                except Exception:
                    pass
    print(f"Resuming – {len(done_set)} entries already completed.")
else:
    print("Starting fresh (no existing output found).")

In [ ]:
# ==== Cell 10: Main Generation Loop ====
# Deterministic: do_sample=False; no temperature/top_p/top_k passed.

MAX_NEW = MAX_NEW_TOKENS   # single source of truth

out_fh    = open(OUT_JSONL,    "a", encoding="utf-8")
errors_fh = open(ERRORS_JSONL, "a", encoding="utf-8")

try:
    for doc_id, doc_df in df.groupby("doc_id", sort=False):
        doc_df = doc_df.sort_values("pid").reset_index(drop=True)

        for idx, row in doc_df.iterrows():
            pid      = int(row["pid"])
            text_fr  = str(row.get("text_fr", row.get("text", "")))
            weak_hint = str(row.get("weak_role_hint", "unknown"))

            # Skip already completed entries
            if (doc_id, pid) in done_set:
                continue

            # Build context window from preceding rows of the SAME document
            ctx_rows = (
                doc_df[doc_df["pid"] < pid]
                .tail(CTX_N)
                [["pid", "text_fr"] if "text_fr" in doc_df.columns else ["pid", "text"]]
                .rename(columns={"text": "text_fr"})
                .to_dict("records")
            )
            allowed_ctx_pids = {int(r["pid"]) for r in ctx_rows}

            user_prompt = build_user_prompt(doc_id, pid, text_fr, ctx_rows, weak_hint)

            # Safe context budgeting: trim context lines if needed
            user_prompt = safe_context_budget(
                tokenizer, SYSTEM_PROMPT, user_prompt, MAX_MODEL_LEN, MAX_NEW
            )

            messages = [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": user_prompt},
            ]

            try:
                inputs = tokenizer.apply_chat_template(
                    messages,
                    return_tensors="pt",
                    add_generation_prompt=True,
                ).to(model.device)

                with torch.no_grad():
                    gen = model.generate(
                        inputs,
                        max_new_tokens=MAX_NEW,
                        do_sample=False,          # deterministic
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                decoded = tokenizer.decode(
                    gen[0][inputs.shape[-1]:], skip_special_tokens=True
                ).strip()

                # Strip optional markdown code fences
                if decoded.startswith("```"):
                    decoded = re.sub(r"^```[^\n]*\n", "", decoded)
                    decoded = re.sub(r"```$", "", decoded).strip()

                obj = json.loads(decoded)
                ann = normalize_pred(obj, pid, allowed_ctx_pids, text_fr, weak_hint)
                ann = boundary_consistency_check(ann, doc_id, pid)

                record = {
                    "doc_id": doc_id,
                    "pid": pid,
                    "text_fr": text_fr,
                    "annotation": ann,
                }
                out_fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                out_fh.flush()
                done_set.add((doc_id, pid))

            except Exception as e:
                err_record = {
                    "doc_id": doc_id,
                    "pid": pid,
                    "text_fr": text_fr,
                    "error": str(e),
                    "raw": decoded if "decoded" in dir() else "",
                }
                errors_fh.write(json.dumps(err_record, ensure_ascii=False) + "\n")
                errors_fh.flush()
                print(f"  [ERROR] doc={doc_id} pid={pid}: {e}")

finally:
    out_fh.close()
    errors_fh.close()

print(f"Done. {len(done_set)} entries written to {OUT_JSONL}")

In [ ]:
# ==== Cell 11: Two-Pass Error Recovery ====
# Re-runs failed entries by rebuilding context from the coreset dataframe.
# Run this cell AFTER the main loop to attempt recovery of failed paragraphs.

if not ERRORS_JSONL.exists():
    print("No errors file found – nothing to recover.")
else:
    error_rows: list = []
    with open(ERRORS_JSONL, "r", encoding="utf-8") as _ef:
        for _line in _ef:
            if _line.strip():
                try:
                    error_rows.append(json.loads(_line))
                except Exception:
                    pass

    print(f"Recovering {len(error_rows)} failed entries...")

    # Re-load done_set from current output
    done_set_r: set = set()
    if OUT_JSONL.exists():
        with open(OUT_JSONL, "r", encoding="utf-8") as _f:
            for _line in _f:
                if _line.strip():
                    try:
                        _r = json.loads(_line)
                        done_set_r.add((_r["doc_id"], int(_r["pid"])))
                    except Exception:
                        pass

    # Build a lookup: (doc_id, pid) -> row from coreset
    coreset_lookup: dict = {
        (str(row["doc_id"]), int(row["pid"])): row
        for _, row in df.iterrows()
    }
    coreset_by_doc: dict = {
        d: g.sort_values("pid").reset_index(drop=True)
        for d, g in df.groupby("doc_id")
    }

    recovered, still_failed = 0, 0
    recovered_fh = open(OUT_JSONL, "a", encoding="utf-8")
    new_errors_fh = open(ERRORS_JSONL.with_suffix(".pass2.jsonl"), "w", encoding="utf-8")

    try:
        for err in error_rows:
            doc_id = str(err["doc_id"])
            pid    = int(err["pid"])

            if (doc_id, pid) in done_set_r:
                continue   # already recovered in a prior run

            text_fr   = str(err.get("text_fr", ""))
            weak_hint = "unknown"

            # Rebuild context from coreset dataframe
            doc_sub = coreset_by_doc.get(doc_id)
            if doc_sub is None:
                new_errors_fh.write(json.dumps({**err, "error2": "doc not in coreset"}, ensure_ascii=False) + "\n")
                still_failed += 1
                continue

            text_col  = "text_fr" if "text_fr" in doc_sub.columns else "text"
            ctx_rows  = (
                doc_sub[doc_sub["pid"] < pid]
                .tail(CTX_N)[["pid", text_col]]
                .rename(columns={text_col: "text_fr"})
                .to_dict("records")
            )
            allowed_ctx_pids = {int(r["pid"]) for r in ctx_rows}

            user_prompt = build_user_prompt(doc_id, pid, text_fr, ctx_rows, weak_hint)
            user_prompt = safe_context_budget(
                tokenizer, SYSTEM_PROMPT, user_prompt, MAX_MODEL_LEN, MAX_NEW
            )

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ]

            decoded2 = ""
            try:
                inputs2 = tokenizer.apply_chat_template(
                    messages, return_tensors="pt", add_generation_prompt=True
                ).to(model.device)

                with torch.no_grad():
                    gen2 = model.generate(
                        inputs2,
                        max_new_tokens=MAX_NEW,
                        do_sample=False,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                decoded2 = tokenizer.decode(
                    gen2[0][inputs2.shape[-1]:], skip_special_tokens=True
                ).strip()
                if decoded2.startswith("```"):
                    decoded2 = re.sub(r"^```[^\n]*\n", "", decoded2)
                    decoded2 = re.sub(r"```$", "", decoded2).strip()

                obj2 = json.loads(decoded2)
                ann2 = normalize_pred(obj2, pid, allowed_ctx_pids, text_fr, weak_hint)
                ann2 = boundary_consistency_check(ann2, doc_id, pid)

                record2 = {
                    "doc_id": doc_id,
                    "pid":    pid,
                    "text_fr": text_fr,
                    "annotation": ann2,
                }
                recovered_fh.write(json.dumps(record2, ensure_ascii=False) + "\n")
                recovered_fh.flush()
                done_set_r.add((doc_id, pid))
                recovered += 1

            except Exception as e2:
                new_errors_fh.write(
                    json.dumps({**err, "error2": str(e2), "raw2": decoded2}, ensure_ascii=False) + "\n"
                )
                new_errors_fh.flush()
                still_failed += 1
    finally:
        recovered_fh.close()
        new_errors_fh.close()

    print(f"Recovery complete: {recovered} recovered, {still_failed} still failed.")

In [ ]:
# ==== Cell 12: Quality Report ====

rows_all: list = []
if OUT_JSONL.exists():
    with open(OUT_JSONL, "r", encoding="utf-8") as _f:
        for _line in _f:
            if _line.strip():
                try:
                    rows_all.append(json.loads(_line))
                except Exception:
                    pass

print(f"Total annotated paragraphs: {len(rows_all)}")

type_counts: dict = defaultdict(int)
rel_counts:  dict = defaultdict(int)
conf_counts: dict = defaultdict(int)
review_count = 0

for r in rows_all:
    ann = r.get("annotation", {})
    type_counts[ann.get("type", "unknown")] += 1
    conf_counts[ann.get("model_confidence", "C")] += 1
    if ann.get("_review_issues"):
        review_count += 1
    for _pid, rels in ann.get("matched_paras", {}).items():
        if isinstance(rels, list):
            for rel in rels:
                rel_counts[rel] += 1
        elif isinstance(rels, str):
            rel_counts[rels] += 1

print("\n--- Type Distribution ---")
for k, v in sorted(type_counts.items()):
    print(f"  {k}: {v}")

print("\n--- Relation Distribution ---")
for k, v in sorted(rel_counts.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

print("\n--- Confidence Distribution ---")
for k, v in sorted(conf_counts.items()):
    print(f"  {k}: {v}")

print(f"\nParagraphs flagged for review: {review_count}")
print(f"Review queue size: {len(REVIEW_QUEUE)}")

In [ ]:
# ==== Cell 13: Submission Exporter (M3) – Per-Doc JSON ====

def flatten_matched_paras_for_submit(mp_internal: dict) -> dict:
    """Convert internal {pid: [rel,...]} to submission {pid: 'rel'} format."""
    out: dict = {}
    if not isinstance(mp_internal, dict):
        return out
    for pid, rels in mp_internal.items():
        if isinstance(rels, str):
            rel = rels
        elif isinstance(rels, list):
            rel = flatten_rel_list([r for r in rels if r in VALID_REL])
        else:
            rel = None
        if rel in VALID_REL:
            out[str(int(pid))] = rel
    return out


def doc_level_think(preambular_paras: list, operative_paras: list) -> str:
    return (
        f"Types assigned using French triggers; "
        f"preambular_paras={len(preambular_paras)}, "
        f"operative_paras={len(operative_paras)}. "
        f"Relations decided by whether scope/conditions are modified (modifying) "
        f"or only parallel info is added (complemental)."
    )


def export_submission(clean_jsonl_path: Path, out_dir: Path) -> None:
    """Write one JSON file per document to *out_dir*."""
    out_dir.mkdir(parents=True, exist_ok=True)

    rows: list = []
    with open(clean_jsonl_path, "r", encoding="utf-8") as _f:
        for _line in _f:
            if _line.strip():
                try:
                    rows.append(json.loads(_line))
                except Exception:
                    pass

    by_doc: dict = defaultdict(list)
    for r in rows:
        by_doc[r["doc_id"]].append(r)

    for doc_id, items in by_doc.items():
        items = sorted(items, key=lambda x: int(x["pid"]))

        pre_list: list = []
        op_list:  list = []
        paras_out: list = []

        for r in items:
            ann = r["annotation"]
            pid = int(r["pid"])

            if ann["type"] == "preambular":
                pre_list.append(pid)
            else:
                op_list.append(pid)

            paras_out.append({
                "para_number":   pid,
                "para":          r["text_fr"],
                "type":          ann["type"],
                "tags":          ann.get("tags", []),
                "matched_paras": flatten_matched_paras_for_submit(
                    ann.get("matched_paras", {})
                ),
                "think":         ann.get("think", ""),
            })

        doc_obj = {
            "TEXT_ID": doc_id,
            "METADATA": {
                "structure": {
                    "preambular_paras": sorted(pre_list),
                    "operative_paras":  sorted(op_list),
                    "think":            doc_level_think(pre_list, op_list),
                    "nb_paras":         len(items),
                }
            },
            "body": {
                "paras": sorted(paras_out, key=lambda x: x["para_number"])
            },
        }

        with open(out_dir / f"{doc_id}.json", "w", encoding="utf-8") as _out:
            json.dump(doc_obj, _out, ensure_ascii=False, indent=2)

    print(f"Exported {len(by_doc)} doc(s) to {out_dir}")


# Run the exporter
export_submission(OUT_JSONL, SUBMIT_DIR)
print("Sample doc IDs:", list(SUBMIT_DIR.glob("*.json"))[:5])

In [ ]:
# ==== Cell 14: Package Submission ZIP ====
# Compress all per-doc JSON files into a single ZIP for upload.

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as _zf:
    for _json_file in sorted(SUBMIT_DIR.glob("*.json")):
        _zf.write(_json_file, arcname=_json_file.name)

print(f"ZIP created: {ZIP_PATH}")
print(f"Archived {len(list(SUBMIT_DIR.glob('*.json')))} JSON file(s).")